# VISHC-PPD segmentation benchmark — Google Colab

Trains and evaluates the five benchmark models of
*A Vietnamese child and adult plantar pressure image dataset with eight-region annotations for foot pathology*.

| Step | What it does |
|---|---|
| 0 | Mount Drive, install dependencies |
| 1 | Validate the dataset copy |
| 2 | Train the five models × `SEEDS` (one loop cell; finished runs are skipped) |
| 3 | Evaluate all checkpoints on the test split |
| 4 | Export predictions of the best-performing model (masks + LabelMe JSON) |

Expected Drive layout (`PROJECT_ROOT` in the path cell must point to it):
```
MyDrive/VISHC-PPD/
├── VISHC-PPD-dataset/              ← the released dataset
└── VISHC-PPD-segmentation-models/  ← this code repository
```
Runtime → Change runtime type → **GPU** (T4 works; L4 / A100 are faster).

## Step 0 — Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# ── paths (adjust PROJECT_ROOT if the folder lives elsewhere) ──────────────────────────────────────────────
PROJECT_ROOT = '/content/drive/MyDrive/VISHC-PPD'
DATASET_ROOT = f'{PROJECT_ROOT}/VISHC-PPD-dataset'
CODE_DIR     = f'{PROJECT_ROOT}/VISHC-PPD-segmentation-models'

import os
for name, p in [('DATASET_ROOT', DATASET_ROOT), ('CODE_DIR', CODE_DIR)]:
    print(('OK  ' if os.path.exists(p) else 'MISSING  ') + name + ' = ' + p)
os.chdir(CODE_DIR)
print('cwd:', os.getcwd())

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU — select a GPU runtime'
!pip install -q -r requirements.txt
import torch, transformers, timm, albumentations, shapely
print('torch', torch.__version__, '| transformers', transformers.__version__, '| timm', timm.__version__)

## Step 1 — Validate the dataset copy

In [ ]:
!python validate_dataset.py --dataset_root "$DATASET_ROOT"

## Step 2 — Training
Identical protocol for each of the five models: 60 epochs, batch 8, AdamW (head LR 3e-4, pretrained encoder LR 3e-5), 3 warm-up epochs then cosine decay, 512×256 input, encoder frozen for the first 3 epochs, subject-level validation hold-out, best checkpoint by validation mIoU.

After a disconnect, re-running Step 0 and Step 2 resumes the benchmark: runs already on Drive are skipped automatically. A run on Drive is reused only if its `args.json` records the same `package_version` as the installed code and the same training pool (`train_pool`: number and identity of the training images); any other run is trained again, so results always come from this code on this dataset.

In [ ]:
SEEDS  = [42, 43, 44]               # three seeds -> mean ± SD in the paper ([42] alone gives a quick first pass)
MODELS = ['deeplabv3_r50', 'unet', 'segformer_b2', 'swin_unet', 'dinov2_vitb14']
EXTRA  = {}                         # identical protocol for every model; per-model overrides go here if ever needed

# Copy the dataset to the local SSD once (Drive I/O is slow)
LOCAL_DATA = '/content/vishc_ppd_data'
if not os.path.exists(LOCAL_DATA):
    !cp -r "$DATASET_ROOT" "$LOCAL_DATA"
print('data at', LOCAL_DATA)

In [ ]:
import subprocess, shlex, shutil, os, json
from vishc_ppd import __version__ as CODE_VERSION

LOCAL_RUNS = '/content/runs'                 # fast local disk (lost when the session ends)
DRIVE_RUNS = f'{CODE_DIR}/runs'              # persistent: best.pt + logs are copied here after every run

import hashlib
from vishc_ppd.data import collect_from_dataset
_pool = collect_from_dataset(LOCAL_DATA, 'train')
TRAIN_POOL = f"{len(_pool)}:{hashlib.sha1(chr(10).join(sorted(s.image_id for s in _pool)).encode()).hexdigest()[:12]}"

def drive_run_is_current(out_drive):
    """A run on Drive is reused only if it was produced by this version of the code on this training pool."""
    args_file = f'{out_drive}/args.json'
    if not (os.path.exists(f'{out_drive}/best.pt') and os.path.exists(args_file)):
        return False
    a = json.load(open(args_file))
    return a.get('package_version') == CODE_VERSION and a.get('train_pool') == TRAIN_POOL

def train(model, seed, extra=''):
    out_local = f'{LOCAL_RUNS}/{model}/seed{seed}'
    out_drive = f'{DRIVE_RUNS}/{model}/seed{seed}'
    if drive_run_is_current(out_drive):
        print('skip (already on Drive):', out_drive)
        if not os.path.exists(f'{out_local}/best.pt'):
            shutil.copytree(out_drive, out_local, dirs_exist_ok=True)
        return
    cmd = f'python train.py --model {model} --dataset_root "{LOCAL_DATA}" --out_dir "{out_local}" --seed {seed} {extra}'
    print(cmd); subprocess.run(shlex.split(cmd), check=True)
    os.makedirs(out_drive, exist_ok=True)
    for f in ('best.pt', 'metrics.json', 'args.json', 'split.json'):   # last.pt stays local (saves Drive space)
        shutil.copy2(f'{out_local}/{f}', f'{out_drive}/{f}')
    print('saved to Drive:', out_drive)

for m in MODELS:
    for s in SEEDS:
        train(m, s, EXTRA.get(m, ''))

## Step 3 — Evaluation on the test split

In [ ]:
!python evaluate.py --dataset_root "$LOCAL_DATA" --runs_dir "$LOCAL_RUNS" --out_dir evaluation
import pandas as pd
pd.read_csv('evaluation/metrics_summary.csv')

In [ ]:
pd.read_csv('evaluation/metrics_per_region.csv')

## Step 4 — Export predictions (masks + LabelMe JSON) for the best-performing model
The model with the highest mean test mIoU in `evaluation/metrics_summary.csv` is selected automatically, and its best seed (highest test mIoU in `evaluation/metrics_runs.csv`) is used for the export. Set `MODEL` / `SEED` by hand to export another checkpoint.

In [ ]:
import pandas as pd

summary = pd.read_csv('evaluation/metrics_summary.csv')
runs    = pd.read_csv('evaluation/metrics_runs.csv')
summary['miou_mean'] = summary['miou'].astype(str).str.split('±').str[0].astype(float)
best  = summary.sort_values('miou_mean', ascending=False).iloc[0]
MODEL = best['model']
SEED  = int(runs[runs['model'] == MODEL].sort_values('miou', ascending=False).iloc[0]['run'].replace('seed', ''))
print(f"Best model: {best['display']} ({MODEL}), mean test mIoU {best['miou']} -> exporting seed {SEED}")

!python predict.py --checkpoint $LOCAL_RUNS/$MODEL/seed$SEED/best.pt --dataset_root "$LOCAL_DATA" --split test --out_dir preds/$MODEL --save_json
!ls preds/$MODEL | head

## Step 5 — Keep the results
`best.pt`, `metrics.json`, `args.json`, `split.json` of every run are already on Drive (`VISHC-PPD-segmentation-models/runs/`). `evaluation/` and `preds/` are written into the code folder on Drive as well. The zip below bundles everything except checkpoints:

In [ ]:
!cd "$CODE_DIR" && zip -q -r /content/vishc_ppd_results.zip evaluation preds runs -x "*.pt"
from google.colab import files
files.download('/content/vishc_ppd_results.zip')